# Integrated Multi-Modal Spatial Biology & Metabolomics Pipeline
### Publication-Quality Analytics & Visualization Suite

This refactored notebook provides a streamlined interface for statistical analysis and visualization of multi-modal tissue datasets (SRS, CODEX, MALDI dual-mode mass spectrometry, collagen SHG morphology, and snRNA-seq).

#### Available Plotting Tools:
1. **Statistical Group Comparisons**: Violin / Box + Strip Plots + Welch's ANOVA & Games-Howell post-hoc tests with significance brackets.
2. **Paired Multi-Feature Bar Charts**: Multi-species bar charts with error bars (Std/SEM) and automated significance stars.
3. **Compact Inset Bar Plots**: Publication-ready transparent figure insets for anatomical overlays.
4. **Correlation Analysis**: Metabolite correlation matrix heatmaps & target metric correlation rank plots.
5. **Spatial Coordinates & Distributions**: Centroid tissue mapping & stacked cell composition charts.
6. **UMAP & Dimensionality Reduction**: Categorical cluster projections & continuous gradient heatmaps.
7. **Dataset Registry & Audit Logger**: Automatic multi-project dataset switching and session audit trail tracking.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root and sk_classifier to sys.path
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

# Import unified plotting and statistical toolkit
from ip_plotting import (
    DATASET_REGISTRY,
    DATASET_LOAD_LOG,
    load_project_dataset,
    display_dataset_log,
    UNIT_CLUSTERS,
    CLUSTER_COLORS,
    FTU_MAP,
    FIB_MAP,
    GROUP_ORDER,
    UNIT_ORDER,
    METRIC_GROUPS,
    plot_group_comparisons,
    plot_feature_bars,
    plot_compact_inset,
    plot_correlation_matrix,
    plot_target_correlations,
    plot_spatial_coordinates,
    plot_category_distribution,
    plot_umap_embedding,
    run_welch_games_howell,
    stars,
    format_p_value
)

# Set global plotting aesthetic
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Helvetica']
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 1.2
print("✅ Setup completed: Toolkit and styling initialized.")

## 1. Project Dataset Registry & Session Audit Log
Select the active project below (`CODEX_WSI`, `VALLON_D2O_SGLT2I`, `COLLAGEN_PROXIMITY`, or `SNRNA_INTEGRATION`). 
All loaded files are automatically audited and logged with timestamps, file paths, and record dimensions.

In [ ]:
# Choose active project to load
ACTIVE_PROJECT = "CODEX_WSI"  # Options: 'CODEX_WSI' | 'VALLON_D2O_SGLT2I' | 'COLLAGEN_PROXIMITY' | 'SNRNA_INTEGRATION'

if ACTIVE_PROJECT == "CODEX_WSI":
    # Load main CODEX cluster and WSI multi-modal quantification dataset
    # Provide fallback relative path if OneDrive path differs
    df = load_project_dataset(
        "CODEX_WSI", 
        "main_output", 
        fallback_path="./WSI_data/CODEX_clusters-regions_masks_output_csv-test/output.csv"
    )
elif ACTIVE_PROJECT == "VALLON_D2O_SGLT2I":
    # Load Vallon SGLT2i kidney turnover & SRS metrics
    df_srs = load_project_dataset("VALLON_D2O_SGLT2I", "srs_metrics_csv")
    df_to = load_project_dataset("VALLON_D2O_SGLT2I", "turnover_parquet")
    to_cols = ['Image Name', 'ROI Inst', 'Label', 'Protein_Turnover Mean', 'Lipid_Turnover Mean', 'Lipid_to_Protein Mean', 'Lipid_Unsaturation Mean']
    df = pd.merge(df_srs, df_to[to_cols], on=['Image Name', 'ROI Inst', 'Label'], how='left')

# Display dataset audit log
display_dataset_log()

## 2. Data Preprocessing & Anatomical Taxonomy Mapping
Cleans raw outputs, maps clusters to canonical functional units (FTUs), groups metadata, and applies quality-control instance thresholds.

In [ ]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 1. Apply Canonical Taxonomy Mappings
if 'mask_name' in df.columns:
    df['unit_display'] = df['mask_name'].map(UNIT_CLUSTERS).fillna(df['mask_name'])
    df['FTU'] = df['mask_name'].map(FTU_MAP).fillna(df['mask_name'])

if 'group' in df.columns:
    # Standardize region naming
    df = df.replace({
        'Middle Cortex': 'Outer Cortex',
        'Juxtamedullary Cortex': 'Inner Cortex'
    })
    df['fib'] = df['group'].map(FIB_MAP).fillna(df['group'])

# 2. Calculate Group Representation & Apply Minimum Size Filter
if 'group' in df.columns and 'mask_name' in df.columns:
    unit_sizes = df.groupby('mask_name')['mask_name'].transform('count')
    unit_group_sizes = df.groupby(['group', 'mask_name'])['mask_name'].transform('count')
    df['cluster_group_pct'] = (unit_group_sizes / unit_sizes) * 100
    
    # Filter out clusters with < 1% representation within the unit for clean statistics
    filtered_df = df[df['cluster_group_pct'] >= 1.0].copy()
else:
    filtered_df = df.copy()

print(f"Data ready for visualization: {filtered_df.shape[0]:,} instances remaining across {filtered_df.shape[1]} features.")

## 3. Tool 1: Statistical Group Comparisons (Box/Violin + Strip Plots)
Runs Welch's ANOVA and Games-Howell post-hoc tests with automatic significance brackets, mean value callouts, and strip plots.

In [ ]:
# Select metrics to analyze
comparison_metrics = [
    'Redox', 
    'Lipid_Unsaturation', 
    'Lipid_to_Protein', 
    'CL 72:8 (-H)', 
    'CL 74:11 (-H)', 
    'SM 18:1;2O/16:0 (-CH3)', 
    'PS 18:0_18:1 (-H)'
]
available_metrics = [m for m in comparison_metrics if m in filtered_df.columns]

# Run statistical comparison across cortical / medullary zones for Glomeruli (Cluster1) or PT (Cluster2)
glom_subset = filtered_df[filtered_df['mask_name'] == 'Cluster1'] if 'mask_name' in filtered_df.columns else filtered_df

anova_results, posthoc_results = plot_group_comparisons(
    df=glom_subset,
    value_cols=available_metrics,
    group_cols=['group'],
    col_order={'group': ['Outer Cortex', 'Inner Cortex']},
    add_brackets=True,
    palette=['#c75b5b', '#911c1c'],
    output_dir="./plots/glom_regional_comparisons"
)

# Display Top Significant Comparisons
if not posthoc_results.empty:
    print("Significant Pairwise Post-Hoc Differences:")
    display(posthoc_results[posthoc_results['pval'] < 0.05][['item', 'A', 'B', 'mean(A)', 'mean(B)', 'diff', 'pval_formatted']])

## 4. Tool 2: Paired Multi-Feature Bar Charts with Error Bars & Significance
Generates publication-ready paired bar plots comparing multiple species (e.g. Cardiolipins, Sphingomyelins, Redox, Lipid Unsaturation) across anatomical compartments.

In [ ]:
feature_subset = [
    'CL 72:8 (-H)', 
    'CL 74:11 (-H)', 
    'Redox', 
    'SM 18:1;2O/16:0 (-CH3)', 
    'PS 18:0_18:1 (-H)', 
    'SHexCer 42:1;3O (-H)', 
    'SHexCer 42:2;3O (-H)', 
    'Lipid_to_Protein', 
    'Lipid_Unsaturation'
]

# P-values dictionary from post-hoc or prior testing
pval_map = {
    'CL 72:8 (-H)': 0.0001, 
    'CL 74:11 (-H)': 0.0209,
    'Redox': 0.00005, 
    'SM 18:1;2O/16:0 (-CH3)': 0.00028,
    'PS 18:0_18:1 (-H)': 0.00001,
    'SHexCer 42:1;3O (-H)': 0.000128, 
    'SHexCer 42:2;3O (-H)': 0.018, 
    'Lipid_to_Protein': 0.06,
    'Lipid_Unsaturation': 0.02 
}

display_labels = {
    'CL 72:8 (-H)': 'CL 72:8',
    'CL 74:11 (-H)': 'CL 74:11',
    'Redox': 'ORR',
    'SM 18:1;2O/16:0 (-CH3)': 'SM 34:1;2O',
    'PS 18:0_18:1 (-H)': 'PS 36:1',
    'SHexCer 42:1;3O (-H)': 'SHexCer\n42:1;3O',
    'SHexCer 42:2;3O (-H)': 'SHexCer\n42:2;3O',
    'Lipid_to_Protein': 'Lipid/\nProtein',
    'Lipid_Unsaturation': 'Lipid Unsat.'
}

fig, ax = plot_feature_bars(
    df=filtered_df[filtered_df['mask_name'] == 'Cluster1'] if 'mask_name' in filtered_df.columns else filtered_df,
    features=feature_subset,
    group_col='group',
    groups=['Inner Cortex', 'Outer Cortex'],
    pvals=pval_map,
    feature_display_names=display_labels,
    colors=['#c75b5b', '#911c1c'],
    yerr='std',
    save_path="./plots/glom_paired_feature_bars.png"
)
plt.show()

## 5. Tool 3: Compact Inset Bar Plots
Generates compact, borderless square figures with error bars tailored for seamless overlay onto tissue spatial microscopy maps.

In [ ]:
inset_features = [
    'CL 72:8 (-H)', 
    'CL 74:11 (-H)',
    'Redox', 
    'SM 18:1;2O/16:0 (-CH3)',
    'PS 18:0_18:1 (-H)',
    'SHexCer 42:1;3O (-H)', 
    'SHexCer 42:2;3O (-H)'
]
valid_inset_feats = [f for f in inset_features if f in filtered_df.columns]
glom_data = filtered_df[filtered_df['mask_name'] == 'Cluster1'] if 'mask_name' in filtered_df.columns else filtered_df

# 1. Inner Cortex Inset
plot_compact_inset(
    df=glom_data,
    features=valid_inset_feats,
    group_name='Inner Cortex',
    group_col='group',
    ylim=(-2, 5),
    yticks=[-2, 0, 2, 4],
    save_filename="./plots/inner_cortex_gloms_inset.png"
)

# 2. Outer Cortex Inset
plot_compact_inset(
    df=glom_data,
    features=valid_inset_feats,
    group_name='Outer Cortex',
    group_col='group',
    ylim=(-2, 5),
    yticks=[-2, 0, 2, 4],
    save_filename="./plots/outer_cortex_gloms_inset.png"
)
plt.show()

## 6. Tool 4: Correlation Matrix Heatmaps & Target Metric Ranking
Computes pairwise metabolite correlations across functional units and ranks markers correlating with fibrosis/collagen or oxidative stress.

In [ ]:
# 1. Full Correlation Matrix Heatmap
corr_matrix, fig_corr, ax_corr = plot_correlation_matrix(
    df=glom_data,
    features=valid_inset_feats,
    method='spearman',
    annot=True,
    title="Glomeruli Metabolite Correlations",
    figsize=(8, 7),
    save_path="./plots/glom_correlation_heatmap.png"
)
plt.show()

# 2. Ranked Feature Correlations against Target Metric (e.g. Redox or Collagen_Thickness)
target_metric = 'Redox' if 'Redox' in filtered_df.columns else valid_inset_feats[0]
corr_ranking_df, fig_rank, ax_rank = plot_target_correlations(
    df=filtered_df,
    target_metric=target_metric,
    method='spearman',
    top_n=16,
    title=f"Top Features Correlating with {target_metric}",
    save_path=f"./plots/{target_metric}_correlations_ranking.png"
)
plt.show()

## 7. Tool 5: Spatial Coordinate Mapping & Instance Distributions
Visualizes tissue instance centroid coordinates across tissue sections and plots compositional distributions across anatomical regions.

In [ ]:
# 1. Spatial Tissue Coordinate Mapping
if 'centroid_x' in filtered_df.columns and 'centroid_y' in filtered_df.columns:
    plot_spatial_coordinates(
        df=filtered_df,
        x_col='centroid_x',
        y_col='centroid_y',
        hue_col='FTU' if 'FTU' in filtered_df.columns else 'group',
        title="Spatial Instance Centroid Distribution",
        s=12,
        save_path="./plots/spatial_instance_distribution.png"
    )
    plt.show()

# 2. Compositional Breakdown Across Cortical Zones
if 'FTU' in filtered_df.columns and 'group' in filtered_df.columns:
    plot_category_distribution(
        df=filtered_df,
        cat_col='FTU',
        group_col='group',
        normalize=True,
        save_path="./plots/ftu_composition_per_group.png"
    )
    plt.show()

## 8. Tool 6: High-Dimensional UMAP Projections
Renders UMAP coordinates color-coded by categorical functional unit / cluster annotations or continuous metabolite intensities.

In [ ]:
# Check if precomputed UMAP columns exist or load precomputed embeddings
umap_cols = [c for c in ['umap_1', 'umap_2', 'UMAP_1', 'UMAP_2'] if c in filtered_df.columns]

if len(umap_cols) >= 2:
    u1, u2 = umap_cols[0], umap_cols[1]
    
    # Categorical Projection by FTU / Cluster
    plot_umap_embedding(
        df=filtered_df,
        umap_1_col=u1,
        umap_2_col=u2,
        hue_col='FTU' if 'FTU' in filtered_df.columns else 'group',
        palette='tab20',
        title="UMAP: Functional Units & Clusters",
        save_path="./plots/umap_categorical_projection.png"
    )
    plt.show()
    
    # Continuous Metabolic Gradient (e.g. Redox / ORR)
    if 'Redox' in filtered_df.columns:
        plot_umap_embedding(
            df=filtered_df,
            umap_1_col=u1,
            umap_2_col=u2,
            hue_col='Redox',
            is_continuous=True,
            cmap='magma',
            title="UMAP: Redox Ratio Mean (ORR)",
            save_path="./plots/umap_redox_gradient.png"
        )
        plt.show()
else:
    print("ℹ️ Note: To generate new UMAP embeddings from raw features, use sk_classifier/umap_analysis.py generate_ion_umaps().")

## 9. Session Summary & Audit Trail
Review all datasets loaded, transformed, and visualized in the current session.

In [ ]:
print("📊 Session Dataset Audit Trail:")
display_dataset_log()